# Prediction Pathway Analysis

Trace how predictions form: buildup, attribution, confidence,
competition, and commit point.

## Why This Matters

Path prediction pathway traces specific computational pathways through the network. Path-level analysis reveals how information flows from specific input tokens through specific components to specific output predictions, providing the most detailed view of model computation.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Conmy et al. (2023) "Towards Automated Circuit Discovery"](https://arxiv.org/abs/2304.14997) — Automated methods for circuit finding via edge pruning

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_pathway_analysis import (
    prediction_buildup, prediction_component_attribution,
    prediction_confidence_evolution, prediction_competition,
    prediction_commit_point,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Prediction Buildup

How does the predicted token evolve layer by layer?

In [ ]:
result = prediction_buildup(model, tokens, position=3)
print(f"Target token: {result['target_token']}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: top={p['top_prediction']:3d} "
          f"(logit={p['top_logit']:.3f}), target_rank={p['target_rank']}, "
          f"target_prob={p['target_prob']:.4f}")

## Component Attribution

How much does attention vs MLP contribute to the target logit?

In [ ]:
result = prediction_component_attribution(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: attn={p['attn_logit_contribution']:+.4f}, "
          f"mlp={p['mlp_logit_contribution']:+.4f}, "
          f"total={p['total_contribution']:+.4f}")

## Confidence Evolution

How confident is the model at each layer?

In [ ]:
result = prediction_confidence_evolution(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['max_prob'] * 40)
    print(f"  Layer {p['layer']}: max_prob={p['max_prob']:.4f}, "
          f"H={p['entropy']:.3f}, top={p['top_token']:3d} {bar}")

## Competition

Which tokens are competing at each layer?

In [ ]:
result = prediction_competition(model, tokens, top_k=5)
for p in result['per_layer']:
    cands = ', '.join(f'tok{c["token_id"]}({c["probability"]:.3f})' for c in p['candidates'][:3])
    print(f"  Layer {p['layer']}: margin={p['margin']:.4f} | {cands}")

## Commit Point

At which layer does the model irreversibly commit to its final prediction?

In [ ]:
result = prediction_commit_point(model, tokens)
print(f"Final prediction: token {result['final_prediction']}")
print(f"Commit layer: {result['commit_layer']}\n")
for p in result['per_layer']:
    match = '✓' if p['matches_final'] else '✗'
    print(f"  Layer {p['layer']}: top={p['top_token']:3d} [{match}]")